# Praziquantel drug response of Kenyan field adult parasites

We investigated the drug response to praziquantel (PZQ) of adult male worms from hotspot and non-hotspot villages. These villages show significant difference in schistosome infection prevalence despite program of mass drug administration ([Olilah *et al.*, 2026](https://doi.org/10.1016/j.crpvbd.2026.100349)). One hypothesis to explain this difference is the presence of resistant worms. To test this, we collected larvae from patients, infected snails and then hamsters to generate adult worms. The worms were collected by perfusion and culture for 5 days in bulk. On day 2, worms were exposed to PZQ (1 µg/mL) along with a controls exposed to DMSO only. On day 5, treated worms were observed and all moving worms as well non moving worms were plated individually for a total of 50 worms. In addtion, 10 control worms were also plated. The movement of the worms were recorded on video and analyzed using our Single Worm Analysis Movement Pipeline (SWAMP).

The SWAMP assay was validated in our laboratory using mixtures of SmLE-PZQ-ES and SmLE-PZQ-ER worms. These populations have been purified by marker assisted selection and show more than 377-fold difference in their response to PZQ ([Clec’h *et al.*, 2021](https://www.science.org/doi/10.1126/scitranslmed.abj9114)). We made pools of 100 adult male parasites containing SmLE-PZQ-ES and SmLE-PZQ-ER worms in proportions 99:1 and 95:5. The worms were cultured and treated in bulk similarly to the field samples At day 5, we plated all worms individually and quantified their movement.

## Environment and data

### Environment

Creating a conda environment improves reproducibility by installing specific package and tool versions.

In [ ]:
conda env create -f .env/env.yml

The cells below must be run each time a new Jupyter session is run or when the kernel is rebooted.

In [ ]:
# Activate the environment
source $(sed "s,/bin/conda,," <<<$CONDA_EXE)/etc/profile.d/conda.sh
conda activate pzq_resp

# Remove potential variable interferences
export PERL5LIB=""
export PYTHONNOUSERSITE=1

Below are variables that are used in throughout the notebook.

In [ ]:
# Directories
vdir="data/videos/"
statdir="status/"
resdir="results/"
bin=".bin/"

### Software

Worm movement is quantified from the videos using the SWAMP pipeline. It is available from GitHub.

In [ ]:
[[ ! -d "$bin" ]] && mkdir -p "$bin"

In [ ]:
git clone --recursive https://github.com/fdchevalier/SWAMP "$bin/swamp"

In [ ]:
# Link the pipeline in the conda environment bin folder
ln -sf "$PWD/$bin/SWAMP/swamp" "$CONDA_PREFIX/bin/"

### Worm video data

The video recordings of the worm in 96 well plates is available on the BioImage repository. These videos are those taken in the lab for validation of the SWAMP assay and those from the phenotyping of field worms. They will be downloaded and organized for the SWAMP analysis.

In [ ]:
# Create storage folder
vdir_l="$vdir/lab"
[[ ! -d "$vdir_l" ]] && mkdir -p "$vdir_l"

In [ ]:
root="https://ftp.ebi.ac.uk/pub/databases/biostudies/S-BIAD/516/S-BIAD3516/Files/"
j=1

for i in 1 2
do
    while read line
    do
        path=$(cut -f 1 <<< "$line")
        dn=$(dirname "$path")
        fn=$(basename "$path")
        fn_new="Plate $j.MOV"

        # Download
        wget -P "$vdir_l/$dn/" "$root/$path"

        # Renaming
        mv "$vdir_l/$dn/$fn" "$vdir_l/$dn/$fn_new"
        
        ((j++))

    done < <(wget -O - "$root/S-BIAD3516 - study component $i.tsv" | tail -n +2)
done

unset j root

In [ ]:
# Create storage folder
vdir_f="$vdir/field"
[[ ! -d "$vdir_f" ]] && mkdir -p "$vdir_f"

In [ ]:
root="https://ftp.ebi.ac.uk/pub/databases/biostudies/S-BIAD/520/S-BIAD3520/Files/"

for i in Agok Dago Magawa Mumbo Uhanya Uyawi
do
    while read line
    do
        # Download
        wget -P "$vdir_f/$i" "$root/$line"

    done < <(wget -q -O - "$root/S-BIAD3520 - $i study component.tsv" | tail -n +2)
done

## Analysis

### Movement quantification

#### Laboratory validation

To ensure that the SWAMP assay can detect recovering / resistant worms within a population, we mixed SmLE-PZQ-ES and ER populations in two different proportions: 95:5 and 99:1. These population showed more than 377 fold differences in PZQ response and can be genotyped. We collected adult male worms from our lifecycle maintenance and mix them in a petri dish before splitting them a 6 well plate (33 or 34 worms per well) and culture and treat them in bulk for 5 days. On day 4, we plated them individually in a 96 well plate and recorded their movement on day 5. For more details, please refer to the associated manuscript.

After quantifying of movement from the video, we select the 10 more moving worm for genotyping in order to confirm that the most moving worms have resistant genotype.

##### SWAMP analysis

In [ ]:
# PLate layout

for i in "$vdir_l/1pc/Plate 1.MOV" "$vdir_l/5pc/Plate 3.MOV"
do
    fn="$(basename "$i" .MOV)"
    dn="$(dirname "$i")"
    scripts/layout.sh -g Worms Worms Worms Worms Worms Worms -w 14-23 26-35 38-47 50-59 62-71 74-83 -o "$dn/$fn.tsv"
done

for i in "$vdir_l/1pc/Plate 2.MOV" "$vdir_l/5pc/Plate 4.MOV"
do
    fn="$(basename "$i" .MOV)"
    dn="$(dirname "$i")"
    scripts/layout.sh -g Worms Worms Worms Worms -w 14-23 26-35 38-47 50-59 -o "$dn/$fn.tsv"
done

In [ ]:
# Running SWAMP
resdir_l="$resdir/1-SWAMP/lab/"
[[ ! -d "$resdir_l" ]] && mkdir -p "$resdir_l"

while read -r -d '' i
do
    fn=$(basename "$i" .MOV)
    fd=$(dirname "$i")
    cond=$(basename "$fd")
    sbatch -J "$fn" --export=ALL -o "$statdir/swamp/%j.out" -c 1 --mem=10G --wrap="swamp -i \"$i\" -a 5 -s 185 -p \"$resdir_l/$cond/$fn\" -l \"$fd/$fn.tsv\""
done < <(find "$vdir_l" -type f -name "*MOV" -print0 | sort -z)

##### Concatenation

The movement data are then concatenated into a single table.

In [ ]:
# Concatenating results
resdir2_l="$resdir/2-Analysis/lab/"
[[ ! -d "$resdir2_l" ]] && mkdir -p "$resdir2_l"

In [ ]:
output="$resdir2_l/Plate_mean.tsv"

while read -r -d '' i
do
    plate=$(dirname "$i" | sed -r "s|.*/(.*/.*)|\1| ; s|.*([0-9])|\1|")
    pc=$(dirname "$i" | sed -r "s|.*/(.*/.*)|\1| ; s|([0-9]).*|\1|")
    
    [[ -z "$table" ]] && table="$(head -1 "$i" | sed "s/^/Drug\tPC\tPlate\t/")\n"
    
    table+="$(tail -n +2 "$i" | sed "s/^/PZQ\t$pc\t$plate\t/")\n"
    
done < <(find "$resdir_l" -type f -name "Plate_mean.tsv" -print0 | sort -z)

echo -e "$table" | sed "/^$/d" > "$output"
unset table

##### Worm selection and genotype confirmation

The data are analyzed to select to order worms by movement values. The 10 most active worms were selected and genotype to confirm that the most active worms are of the expected resistant genotype while the remaining are of the expected sensitive genotype. See the associated publication for more information.

In [ ]:
# Select worms for genotyping
for i in 1 5
do
    echo "Proportion: $i%"
    awk -v pc=$i '$2 == pc' "$output" | sort -k 6nr
    echo ""
done

In [ ]:
Rscript "scripts/lab_mvt_analysis.R"

<img alt="Movement data from the lab mixture" src="graphs/Fig. 3 - movement_genotype.png" width="300"/>

These data allow us to estimate the sensitivity and specificity of the method.

In [ ]:
Rscript "scripts/sensitivity_pecificity_CI.R"

#### Field populations

We generated worms from hotspot schistosome populations. These hotspots are locations with relatively high prevalence despite mass PZQ administration ([Olilah *et al.*, 2026](https://doi.org/10.1016/j.crpvbd.2026.100349)). We collected miracidia from patients, exposed snails and then infected hamsters with the cercariae produced. We culture the worms similarly to the lab validation experiment (see publication for details). On day 5, 10 control worms (exposed to DMSO) and 50 PZQ-treated worms (any worms moving + additional non-moving worms) were plated. A 3 minutes video was recorded for each plate and analyzed using SWAMP.

##### SWAMP analysis

In [ ]:
# PLate layout
while read -r -d '' i
do
    fn="$(basename "$i" .MOV)"
    dn="$(dirname "$i")"
    scripts/layout.sh -g Control Treated Treated Treated Treated Treated -w 14-23 26-35 38-47 50-59 62-71 74-83 -o "$dn/$fn.tsv"
done < <(find "$vdir_f" -type f -name "*MOV" -print0 | sort -z)

In [ ]:
resdir_f="$resdir/1-SWAMP/field/"
[[ ! -d "$resdir_f" ]] && mkdir -p "$resdir_f"

In [ ]:
while read -r -d '' i
do
    fn=$(basename "$i" .MOV)
    fd=$(dirname "$i")
    cond=$(basename "$fd")
    
    a=5
    s=185
    t=7
    [[ $cond == Magawa ]] && t=20
    [[ $cond == Uhanya ]] && a=2 && s=182
    
    sbatch -J "$fn" --export=ALL -o "$statdir/swamp/%j.out" -c 1 --mem=10G --wrap="swamp -i \"$i\" -a $a -s $s -t $t -p \"$resdir_f/$cond/$fn\" -l \"$fd/$fn.tsv\""
done < <(find "$vdir_f" -type f -name "*MOV" -print0 | sort -z)

**N.B.**: For a few videos, adjustments were needed to obtain reliable data:
- The plate on the video of the Magawa samples was slightly off which disturbed the well recognition. Thresholding had to be increase to correct this.
- Videos taken for Uhanya samples were of 3 minutes long exactly. It is recommended to skip the first 5 seconds to ensure that the plate is not shaking. In this particular case, we only skipped the first 2 seconds.


##### Concatenation

The movement data are then concatenated into a single table.

In [ ]:
# Concatenating results
resdir2_f="$resdir/2-Analysis/field/"
[[ ! -d "$resdir2_f" ]] && mkdir -p "$resdir2_f"

In [ ]:
output="$resdir2_f/Plate_mean.tsv"
md="data/field_plate_metadata.tsv"

while read -r -d '' i
do
    plate=$(dirname "$i" | sed -r "s|.*/(.*/.*)|\1| ; s|.*([0-9])|\1|")
    loc=$(dirname "$i" | sed -r "s|.*/(.*/.*)|\1| ; s|([A-z])/.*|\1|")
    type=$(awk -v loc=$loc -v plate=$plate '$1 == loc && $4 == plate {print $2}' "$md")
    date=$(awk -v loc=$loc -v plate=$plate '$1 == loc && $4 == plate {print $3}' "$md")
    
    [[ -z "$table" ]] && table="$(head -1 "$i" | sed "s/^/Drug\tLocation\tType\tPlate\tDate\t/")\n"
    
    table+="$(tail -n +2 "$i" | sed "s/^/PZQ\t$loc\t$type\t$plate\t$date\t/")\n"
    
done < <(find "$resdir_f" -type f -name "Plate_mean.tsv" -print0 | sort -z)

echo -e "$table" | sed "/^$/d" > "$output"
unset table

##### Graph and statistical analysis

The concatenated data are analyzed to generate graphs and statistics.

In [ ]:
Rscript "scripts/field_mvt_analysis.R"

<img alt="Movement data from field derived adult worms" src="graphs/Fig. 4 - movement_field.png" width="600"/>

## Analysis of snail related traits

Generating the adult worms required infecting snails either raised at KEMRI or sampled in the field. Data related to snail survival and prevalence after exposure to the parasite larvae were recorded. We analyze these data to test if the origin of the snails had an impact on survival or infection.

In [ ]:
Rscript "scripts/snail_surv_inf_analysis.R"

We also estimate how many genotypes could be expected by snails.

In [ ]:
Rscript "scripts/snail_MOI_estimate.R"